# 01 · Custom residual CNN

A CNN designed from scratch for single-channel 150×150 lensing maps (`deeplense/models/cnn.py`).

- **Stride-1 stem**: substructure signatures are only a few pixels wide, so the first layer
  sees full resolution before any downsampling.
- **Five residual stages** (32→512 channels, stride 2 each): 150 → 75 → 38 → 19 → 10 → 5.
  Residual connections with 1×1 projections keep gradients healthy through 11 conv layers.
- **BatchNorm + GELU**, global average pooling, dropout, linear head: ~4.9 M parameters.
- In-model standardisation with training-set pixel statistics.

Training: AdamW (lr 1e-3, wd 0.05), 1 warm-up epoch then cosine decay, D4 augmentation,
best epoch by validation macro AUC.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from deeplense import CLASS_NAMES
from deeplense.utils import get_device, seed_everything

seed_everything(42)
DEVICE = get_device()
DATA_ROOT = ROOT / "data" / "lensing"
RESULTS = ROOT / "results"
print("device:", DEVICE)

In [ ]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# SMOKE = True : a few hundred images, 2 epochs, just to check everything runs (laptop).
# SMOKE = False: full dataset and full recipe (GPU recommended).
# If a full run already exists in results/<run>/ (e.g. from scripts/train.py),
# it is loaded instead of retraining unless RETRAIN = True.
SMOKE = True
RETRAIN = False

In [ ]:
from deeplense.models import LensingCNN
from deeplense.utils import count_params
print(LensingCNN())
print(f"{count_params(LensingCNN()):,} parameters")

## Train (or load)

In [ ]:
from dataclasses import replace
from deeplense.data import DataConfig, build_loaders
from deeplense.metrics import predict
from deeplense.train import fit
from deeplense.utils import load_json
sys.path.insert(0, str(ROOT / "scripts"))
from train import RECIPES

cfg = RECIPES["cnn"]
data_cfg = DataConfig(root=str(DATA_ROOT), num_workers=2)
if SMOKE:
    cfg = replace(cfg, epochs=2)
    data_cfg = replace(data_cfg, train_per_class=300, test_per_class=100)
RUN = "cnn" + ("_smoke" if SMOKE else "")

loaders = build_loaders(data_cfg)
model = LensingCNN()
criterion = None
run_dir = RESULTS / RUN

if (run_dir / "best.pt").exists() and not RETRAIN:
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location="cpu"))
    model.to(DEVICE)
    results = load_json(run_dir / "metrics.json")
    history = load_json(run_dir / "history.json")
    probs, labels = predict(model, loaders["test"], DEVICE)
    print(f"Loaded {run_dir}  (best epoch {results['best_epoch']})")
else:
    model, out = fit(model, loaders, cfg, DEVICE, criterion=criterion, out_dir=RESULTS, run_name=RUN)
    results, history, probs, labels = out, out["history"], out["probs"], out["labels"]

## Evaluation on the held-out test set

In [ ]:
from deeplense.metrics import classification_metrics, plot_confusion, plot_history, plot_roc

plot_history(history, title=RUN); plt.show()

m = classification_metrics(probs, labels)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_roc(probs, labels, title=RUN, ax=axes[0])
plot_confusion(m["confusion_matrix"], title="Test confusion matrix", ax=axes[1])
plt.tight_layout(); plt.savefig(RESULTS / RUN / "roc_confusion.png", dpi=130, bbox_inches="tight"); plt.show()

print(f"Test accuracy {m['accuracy']:.4f} | macro AUC {m['macro_auc']:.4f}")
for k, v in m["auc_per_class"].items():
    print(f"  {k:<20} AUC {v:.4f}")